# 03 — Extended experiments

Experiments that extend notebook 02 for the paper *Accuracy Is Not Enough*: a YouTube split grouped by video, int8
accuracy over three training seeds, a standardised CPU benchmark (repeated on a second processor), and a decomposition of
the TWIFIL scores. Training, export and quantisation code is copied unchanged from notebooks 01 and 02.

| Stage | What | Runtime | Purpose |
|---|---|---|---|
| S | Video-grouped YouTube split (`youtube_grouped`) and video-overlap statistics of the random split | any, 1 min | video-level leakage |
| A | Classical models on the grouped split | CPU, ~10 min | grouped validation |
| G | Transformers on the grouped split (3 seeds); seeds 42 and 87 retrained and saved for TWIFIL (cleaned) and YouTube | **T4 GPU**, ~3.5 h over 1–2 days | grouped validation; int8 across seeds |
| R | Standardised CPU benchmark: every artefact, including ONNX fp32, measured in the same number of identical passes in one session | CPU, ~3.5–4 h | repeatable timing and memory measurement |
| Q | int8 quantisation of the seed-42 and seed-87 models (default, per-channel without and with reduce-range), quality on the test sets | CPU, ~5–6 h | int8 stability across seeds |
| T | Tables: random vs grouped split, int8 across seeds, standardised deployment table, TWIFIL score decomposition | any, 2 min | — |

**How to run.** Keep `RUN_STAGES = "auto"`. On a **T4** runtime the notebook runs S and G and stops G after
`GPU_TIME_BUDGET_H`; run it again the next day until it prints `0 GPU runs to do`. Then switch to a **CPU** runtime and
*Run all*: it runs S, A, R, Q and T (R does not need stage G, so a CPU session can also come first). Every finished unit
is saved to Drive immediately, so any stage can be interrupted and resumed. The results of notebooks 01 and 02 in
`runs/`, `models/`, `bench/` and `reports/` are only read, never changed; new results go to `runs_ext/`, `models_ext/`,
`bench_ext*/` and `reports_ext/`.

**Processor.** int8 kernels behave differently on processors with other instruction-set features, and timing differs
between processor models, so stages R and Q require the processor named in `CPU_MODEL` (default: the Intel Xeon @ 2.20GHz
of the paper, with AVX2 and without AVX-512, VNNI or AMX). If Colab assigns another one, the notebook stops at the start
with *"needs the paper's processor"*: use *Runtime → Disconnect and delete runtime*, connect again and re-run. Stage R
writes to `bench_ext` + `BENCH_TAG` (`bench_ext_xeon/` by default). For the comparison between processors in the paper,
stage R was also run on an AMD EPYC 7B12 with `CPU_MODEL = None` and `BENCH_TAG = ""`.

**Drive space.** The 16 extra-seed models are stored in half precision (≈4 GB in total) and deleted after stage Q.

## 1. Configuration

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/alg_sentiment_benchmark"
RUN_STAGES = "auto"            # "auto": T4 -> "SG"; CPU -> "SARQ". Stage T (tables) always runs.
SMOKE_TEST = False             # True: tiny subsets, 1 epoch, separate *_smoke folders

# ---- transformers (Hugging Face ids) ----
TRANSFORMERS = {
    "dziribert":    {"hf": "alger-ia/dziribert",                      "lr": 3e-5},
    "marbertv2":    {"hf": "UBC-NLP/MARBERTv2",                       "lr": 3e-5},
    "camelbert_da": {"hf": "CAMeL-Lab/bert-base-arabic-camelbert-da", "lr": 3e-5},
    "me5_small":    {"hf": "intfloat/multilingual-e5-small",          "lr": 5e-5},
}
SEEDS = [13, 42, 87]          # the seeds of the paper
EPOCHS = 4                    # max epochs; early stopping on dev macro-F1
PATIENCE = 1
BATCH_SIZE = 32
MAX_LEN_CAP = 128             # max_len = 99th percentile of train token lengths, capped here
WARMUP_RATIO = 0.10
WEIGHT_DECAY = 0.01
GPU_TIME_BUDGET_H = 3.4       # stage B stops starting new runs after this many hours in the session

GPU_TIME_BUDGET_H = 3.4        # stage G stops starting new runs after this many hours in the session
ALLOW_CPU_TRAINING = False     # stage G refuses to train on CPU unless True

# ---- extended experiments ----
GROUPED = "youtube_grouped"    # YouTube split grouped by video (no video in two splits)
GROUPED_SEEDS = [13, 42, 87]
INT8_DATASETS = ["twifil_clean", "youtube"]
EXTRA_SEEDS = [42, 87]         # retrained and saved for the int8 study (seed 13 is the benchmarked model)
DELETE_EXTRA_WEIGHTS_AFTER_USE = True
EVAL_THREADS = 0               # stage Q quality evaluation; 0 = all vCPUs (checked to give identical predictions)

# ---- classical models: hyper-parameter grids (selected on dev macro-F1) ----
CLASSICAL_GRIDS = {
    "nb_char":      {"alpha": [0.01, 0.03, 0.1, 0.3, 1.0]},
    "svm_char":     {"C": [0.03, 0.1, 0.3, 1.0]},
    "lr_word_char": {"C": [0.25, 1.0, 4.0, 16.0, 64.0]},
}

# ---- standardised CPU benchmark (stage R) ----
BENCH_DATASETS = ["twifil_clean", "youtube"]
BENCH_THREADS = 1
BENCH_PASSES = 5               # identical passes for every artefact, interleaved in random order
N_SINGLE = 300                 # texts timed one at a time per round
BENCH_ROUNDS = 3
WARMUP_CALLS = 20
BENCH_BATCH = 32
BENCH_TIMEOUT_S = 3600
CHECK_N = 256                  # first test texts re-predicted to confirm each artefact matches notebook 02
MEASURE_BATCH_MEMORY = False   # True: re-run batch scoring of the full test set per artefact (+4 h); False: use the
                               # main pass of notebook 02, which measured it once, identically, for every artefact
LOCAL_ARTEFACTS = "/content/bench_artefacts"   # artefacts are copied to local disk, so Drive I/O is not timed

# the processor of the paper: AVX2 without AVX-512 or VNNI (int8 behaviour depends on it)
STRICT_CPU = True
CPU_MODEL = "Intel(R) Xeon(R) CPU @ 2.20GHz"   # exact processor name of the paper: stages R and Q refuse any other
                                               # processor, even one with the same flags (None = check the flags only)
BENCH_TAG = "_xeon"            # stage R writes to bench_ext_xeon/; use another tag to keep a run on another processor apart
REQUIRED_FLAGS = ("avx2",)
FORBIDDEN_FLAGS = ("avx512f", "avx512_vnni", "avx_vnni", "amx_int8")
PINNED = {"onnxruntime": "1.30.0", "scikit-learn": "1.6.1"}   # versions used in the paper

SCENARIOS = {
    "Tight budget (≤ 25 MiB, p95 ≤ 20 ms)":      (25, 20),
    "Moderate budget (≤ 150 MiB, p95 ≤ 100 ms)": (150, 100),
    "No constraint":                             (float("inf"), float("inf")),
}
N_BOOTSTRAP = 1000

# ---- names and families used in tables and figures ----
DISPLAY = {"nb_char": "NB-char", "svm_char": "SVM-char", "lr_word_char": "LR-word+char",
           "dziribert": "DziriBERT", "marbertv2": "MARBERTv2", "camelbert_da": "CAMeLBERT-DA", "me5_small": "mE5-small"}
FAMILY = {"nb_char": "Classical (TF-IDF)", "svm_char": "Classical (TF-IDF)", "lr_word_char": "Classical (TF-IDF)",
          "me5_small": "Multilingual small transformer",
          "dziribert": "Arabic/dialect BERT", "marbertv2": "Arabic/dialect BERT", "camelbert_da": "Arabic/dialect BERT"}

## 2. Environment

In [ ]:
import os, sys, json, time, math, shutil, subprocess, platform, random, glob, re, datetime, warnings
SESSION_START = time.time()

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
else:
    PROJECT_DIR = os.environ.get("ALG_PROJECT_DIR", os.path.abspath("alg_sentiment_benchmark"))
    LOCAL_ARTEFACTS = os.environ.get("ALG_LOCAL_ARTEFACTS", os.path.abspath("bench_artefacts"))


def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


# the int8 kernels and the pickled classical models depend on these versions: use the paper's
from importlib import metadata as _md
for _pkg, _ver in PINNED.items():
    try:
        _have = _md.version(_pkg)
    except _md.PackageNotFoundError:
        _have = None
    if _have != _ver:
        print(f"installing {_pkg}=={_ver} (found {_have})")
        _pip(f"{_pkg}=={_ver}")
for _mod, _pkg in (("onnx", "onnx"), ("psutil", "psutil")):
    try:
        __import__(_mod)
    except ImportError:
        _pip(_pkg)

import numpy as np
import pandas as pd
import joblib
import sklearn
from sklearn.metrics import f1_score, accuracy_score

suffix = "_smoke" if SMOKE_TEST else ""
PROC_DIR = os.path.join(PROJECT_DIR, "processed")
CODE_DIR = os.path.join(PROJECT_DIR, "code")
RUNS_DIR = os.path.join(PROJECT_DIR, "runs" + suffix)            # results of notebook 02 (read only)
MODELS_DIR = os.path.join(PROJECT_DIR, "models" + suffix)
BENCH_DIR = os.path.join(PROJECT_DIR, "bench" + suffix)
RUNS_EXT_DIR = os.path.join(PROJECT_DIR, "runs_ext" + suffix)    # results of this notebook
MODELS_EXT_DIR = os.path.join(PROJECT_DIR, "models_ext" + suffix)
BENCH_EXT_DIR = os.path.join(PROJECT_DIR, "bench_ext" + suffix)
BENCH_R_DIR = BENCH_EXT_DIR + BENCH_TAG                          # stage R output (standardised benchmark)
REPORT_EXT_DIR = os.path.join(PROJECT_DIR, "reports_ext" + suffix)
LOCAL_CODE = os.path.join(os.path.dirname(LOCAL_ARTEFACTS.rstrip("/")), "ext_code")
for d in (CODE_DIR, RUNS_EXT_DIR, MODELS_EXT_DIR, BENCH_EXT_DIR, BENCH_R_DIR, REPORT_EXT_DIR, LOCAL_ARTEFACTS, LOCAL_CODE):
    os.makedirs(d, exist_ok=True)
if SMOKE_TEST:
    SEEDS, EPOCHS, GROUPED_SEEDS, EXTRA_SEEDS = SEEDS[:1], 1, GROUPED_SEEDS[:1], EXTRA_SEEDS[:1]
    CLASSICAL_GRIDS = {k: {p: v[:1] for p, v in g.items()} for k, g in CLASSICAL_GRIDS.items()}
    N_SINGLE, WARMUP_CALLS, BENCH_PASSES, CHECK_N = 30, 3, 2, 64

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else None
except ImportError:
    torch, DEVICE, GPU_NAME = None, "cpu", None


def cpu_name():
    try:
        with open("/proc/cpuinfo") as f:
            for line in f:
                if line.startswith("model name"):
                    return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or "unknown"


def all_cpu_flags():
    try:
        with open("/proc/cpuinfo") as f:
            return set(next((l.split(":", 1)[1].split() for l in f if l.startswith("flags")), []))
    except OSError:
        return set()


def cpu_flags():
    """ISA extensions that decide int8 kernel behaviour (VNNI avoids the U8S8 saturation issue)."""
    return [x for x in ("avx2", "avx512f", "avx512_vnni", "avx_vnni", "amx_int8") if x in all_cpu_flags()]


def paper_cpu():
    fl = all_cpu_flags()
    flags_ok = all(f in fl for f in REQUIRED_FLAGS) and not any(f in fl for f in FORBIDDEN_FLAGS)
    return flags_ok and (CPU_MODEL is None or cpu_name() == CPU_MODEL)


VERSIONS = {"python": platform.python_version(), "torch": getattr(torch, "__version__", None),
            "transformers": __import__("transformers").__version__ if torch is not None else None,
            "onnxruntime": __import__("onnxruntime").__version__, "scikit-learn": sklearn.__version__}
print(f"Colab: {IN_COLAB} | device: {DEVICE} {GPU_NAME or ''} | CPU: {cpu_name()} ({os.cpu_count()} vCPU) | "
      f"flags: {cpu_flags()} | same CPU type as the paper: {paper_cpu()}")
print("versions:", VERSIONS)
AUTO_STAGES = RUN_STAGES == "auto"
if AUTO_STAGES:
    RUN_STAGES = "SG" if DEVICE == "cuda" else "SARQ"
print(f"Stages this session: {RUN_STAGES} + T" + (" (auto)" if AUTO_STAGES else ""))
if DEVICE == "cpu" and "R" in RUN_STAGES and STRICT_CPU and not paper_cpu():
    raise RuntimeError(f"Stage R needs the paper's processor ({CPU_MODEL or 'AVX2 without AVX-512/VNNI'}); this session has "
                       f"{cpu_name()} with {cpu_flags()}. Runtime > Disconnect and delete runtime, connect again, and re-run.")
try:
    print(f"Free space on Drive: {shutil.disk_usage(PROJECT_DIR).free / 2**30:.1f} GB")
except OSError:
    pass

### Helper modules
`algsent_utils.py` is the text normalisation of the classical models (identical to notebook 02; the saved models import
it). `bench_worker_ext.py` runs one artefact in a fresh process, in one of two modes: `timing` (cold start, single-text
latency over three rounds of 300 texts after 20 warm-up calls, and the peak memory of that serving process) or `predict`
(predictions for the test texts in batches of 32, in test order, and the peak memory of batch scoring).

In [ ]:
UTILS_SRC = r"""
# Shared text normalisation for the classical (TF-IDF) models.
import re

_URL = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)
_MENTION = re.compile(r"@\w+")
_DIAC = re.compile(r"[ؐ-ًؚ-ٰٟۖ-ۭ]")
_ELONG = re.compile(r"(.)\1{2,}")
_AR = str.maketrans({"أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا", "ى": "ي", "ة": "ه", "ؤ": "و", "ئ": "ي", "ـ": ""})


def normalise(text):
    s = str(text).lower()
    s = _URL.sub(" httpurl ", s)       # kept as a token: official TWIFIL still contains URLs
    s = _MENTION.sub(" @user ", s)
    s = _DIAC.sub("", s).translate(_AR)
    s = _ELONG.sub(r"\1\1", s)
    return " ".join(s.split())


def normalise_batch(texts):
    return [normalise(t) for t in texts]
"""

WORKER_SRC = r"""
# Benchmark worker of notebook 03. One fresh process per call:
#   python bench_worker_ext.py --spec spec.json --texts texts.json --out result.json --mode timing|predict
import argparse, json, os, sys, time

ap = argparse.ArgumentParser()
ap.add_argument("--spec", required=True)
ap.add_argument("--texts", required=True)
ap.add_argument("--out", required=True)
ap.add_argument("--mode", choices=["timing", "predict"], required=True)
ap.add_argument("--threads", type=int, default=1)
ap.add_argument("--n_single", type=int, default=300)
ap.add_argument("--warmup", type=int, default=20)
ap.add_argument("--rounds", type=int, default=3)
ap.add_argument("--batch", type=int, default=32)
ap.add_argument("--limit", type=int, default=0)          # predict mode: only the first N texts (0 = all)
a = ap.parse_args()
for v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[v] = str(a.threads)
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))


def peak_rss_mb():
    # VmHWM: the process's own peak resident set size (reset at exec)
    with open("/proc/self/status") as f:
        for line in f:
            if line.startswith("VmHWM:"):
                return int(line.split()[1]) / 1024
    return float("nan")


def pct(values, p):
    v = sorted(values)
    return v[min(len(v) - 1, int(round(p * (len(v) - 1))))]


spec = json.load(open(a.spec, encoding="utf-8"))
texts = json.load(open(a.texts, encoding="utf-8"))
t0 = time.perf_counter()                            # cold start = library import + model load

if spec["kind"] == "sklearn":
    import joblib
    model = joblib.load(spec["path"])

    def predict(batch):
        return [int(x) for x in model.predict(batch)]

elif spec["kind"] == "torch":
    import torch
    torch.set_num_threads(a.threads)
    torch.set_num_interop_threads(1)
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(spec["path"])
    model = AutoModelForSequenceClassification.from_pretrained(spec["path"]).float().eval()

    def predict(batch):
        enc = tok(batch, truncation=True, max_length=spec["max_len"], padding=True, return_tensors="pt")
        with torch.inference_mode():
            return model(**enc).logits.argmax(-1).tolist()

elif spec["kind"] == "onnx":
    import numpy as np
    import onnxruntime as ort
    from tokenizers import Tokenizer
    so = ort.SessionOptions()
    so.intra_op_num_threads, so.inter_op_num_threads = a.threads, 1
    sess = ort.InferenceSession(spec["path"], so, providers=["CPUExecutionProvider"])
    tk = Tokenizer.from_file(spec["tokenizer"])
    tk.enable_truncation(max_length=spec["max_len"])
    tk.enable_padding(pad_id=spec["pad_id"], pad_token=spec["pad_token"])
    names = [i.name for i in sess.get_inputs()]

    def predict(batch):
        encs = tk.encode_batch(batch)
        feed = {"input_ids": np.array([e.ids for e in encs], dtype=np.int64),
                "attention_mask": np.array([e.attention_mask for e in encs], dtype=np.int64),
                "token_type_ids": np.array([e.type_ids for e in encs], dtype=np.int64)}
        return sess.run(None, {k: feed[k] for k in names})[0].argmax(-1).tolist()
else:
    raise ValueError(spec["kind"])
load_s = time.perf_counter() - t0
out = {"mode": a.mode, "threads": a.threads, "load_s": load_s}

if a.mode == "timing":
    step = max(1, len(texts) // a.n_single)
    sample = texts[::step][: a.n_single]
    for t in sample[: a.warmup]:
        predict([t])
    rounds = []
    for _ in range(a.rounds):
        lat = []
        for t in sample:
            t1 = time.perf_counter()
            predict([t])
            lat.append((time.perf_counter() - t1) * 1000)
        rounds.append({"p50": pct(lat, 0.50), "p95": pct(lat, 0.95), "mean": sum(lat) / len(lat)})
    med = lambda k: pct([r[k] for r in rounds], 0.5)
    out.update({"n_single": len(sample), "rounds": rounds, "p50_ms": med("p50"), "p95_ms": med("p95"),
                "mean_ms": med("mean"), "serving_peak_rss_mb": peak_rss_mb()})
else:
    sel = texts[: a.limit] if a.limit else texts
    preds, w0 = [], time.perf_counter()
    for i in range(0, len(sel), a.batch):
        preds += predict(sel[i:i + a.batch])
    out.update({"n_texts": len(sel), "preds": preds, "wall_s": time.perf_counter() - w0,
                "batch_peak_rss_mb": peak_rss_mb()})
json.dump(out, open(a.out, "w", encoding="utf-8"))
"""

if not os.path.exists(os.path.join(CODE_DIR, "algsent_utils.py")):
    with open(os.path.join(CODE_DIR, "algsent_utils.py"), "w", encoding="utf-8") as f:
        f.write(UTILS_SRC.lstrip())
assert open(os.path.join(CODE_DIR, "algsent_utils.py"), encoding="utf-8").read() == UTILS_SRC.lstrip(), \
    "code/algsent_utils.py differs from notebook 02's version"
for name, src in (("algsent_utils.py", UTILS_SRC), ("bench_worker_ext.py", WORKER_SRC)):
    with open(os.path.join(LOCAL_CODE, name), "w", encoding="utf-8") as f:
        f.write(src.lstrip())
for p in (CODE_DIR, LOCAL_CODE):
    if p not in sys.path:
        sys.path.insert(0, p)
import importlib, algsent_utils
importlib.reload(algsent_utils)
from algsent_utils import normalise, normalise_batch
WORKER = os.path.join(LOCAL_CODE, "bench_worker_ext.py")


def run_worker(spec, texts_path, out, mode, threads=1, extra=()):
    spec_path = out + ".spec.json"
    with open(spec_path, "w") as f:
        json.dump(spec, f)
    cmd = [sys.executable, WORKER, "--spec", spec_path, "--texts", texts_path, "--out", out + ".tmp",
           "--mode", mode, "--threads", str(threads), *map(str, extra)]
    res = subprocess.run(cmd, capture_output=True, text=True, timeout=BENCH_TIMEOUT_S)
    os.remove(spec_path)
    if res.returncode != 0:
        raise RuntimeError(f"worker failed ({mode}, {spec.get('path')}):\n{res.stderr[-2000:]}")
    r = json.load(open(out + ".tmp", encoding="utf-8"))
    os.remove(out + ".tmp")
    return r


print("helper modules ready:", LOCAL_CODE)

## 3. Data

In [ ]:
LABEL_NAMES = {0: "negative", 1: "neutral", 2: "positive", 3: "mixed"}


def load_dataset(ds):
    parts = {}
    for sp in ("train", "dev", "test"):
        df = pd.read_csv(os.path.join(PROC_DIR, ds, f"{sp}.csv"), keep_default_na=False, dtype={"text": str})
        df["label"] = df["label"].astype(int)
        if SMOKE_TEST:
            n = {"train": 240, "dev": 80, "test": 80}[sp]
            df = df.sample(frac=1, random_state=0).groupby("label").head(n // df["label"].nunique() + 1)
            df = df.head(n).reset_index(drop=True)
        parts[sp] = df
    return parts


def scores(y, p):
    y, p = np.asarray(y), np.asarray(p)
    labs = sorted(set(y))
    per = f1_score(y, p, labels=labs, average=None, zero_division=0)
    return {"macro_f1": float(f1_score(y, p, labels=labs, average="macro", zero_division=0)),
            "accuracy": float(accuracy_score(y, p)),
            "per_class_f1": {LABEL_NAMES[l]: float(v) for l, v in zip(labs, per)}}


def run_dir(ds, model, seed, root=None):
    return os.path.join(root or RUNS_DIR, ds, model, f"seed{seed}")


def save_run(ds, model, seed, metrics, test_preds, root=None):
    d = run_dir(ds, model, seed, root)
    os.makedirs(d, exist_ok=True)
    pd.DataFrame({"id": DATA[ds]["test"]["id"], "label": DATA[ds]["test"]["label"], "pred": test_preds}) \
        .to_csv(os.path.join(d, "test_preds.csv"), index=False)
    with open(os.path.join(d, "metrics.json"), "w") as f:      # written last = marks the run as complete
        json.dump(metrics, f, indent=2)


DATA = {}


def load_all():
    for ds in ["twifil_clean", "youtube", "twifil_official", GROUPED]:
        if ds not in DATA and all(os.path.exists(os.path.join(PROC_DIR, ds, f"{sp}.csv")) for sp in ("train", "dev", "test")):
            DATA[ds] = load_dataset(ds)
    global N_LABELS
    N_LABELS = {ds: int(max(p["train"]["label"].max(), p["test"]["label"].max()) + 1) for ds, p in DATA.items()}


load_all()
for ds, p in DATA.items():
    print(f"{ds:16s} train {len(p['train']):6d} | dev {len(p['dev']):5d} | test {len(p['test']):5d}")

## 4. Stage S — video-grouped YouTube split

The random YouTube split is stratified on the five-level label but not grouped by video, so comments on the same video
can fall in training and test data. This stage rebuilds the split with `StratifiedGroupKFold` (10 folds, seed 42; fold 0
= test, fold 1 = development), exactly the grouped option of notebook 01, on the same 44,910 audited comments, and
measures how much the random split mixes videos.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

yt_all = pd.concat([pd.read_csv(os.path.join(PROC_DIR, "youtube", f"{sp}.csv"), keep_default_na=False, dtype=str)
                    for sp in ("train", "dev", "test")]).sort_values("id").reset_index(drop=True)
gdir = os.path.join(PROC_DIR, GROUPED)
if not all(os.path.exists(os.path.join(gdir, f"{sp}.csv")) for sp in ("train", "dev", "test")):
    g = yt_all.copy()
    folds = list(StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
                 .split(g, g["source_label"], groups=g["video"]))
    g["split"] = "train"
    g.loc[folds[0][1], "split"] = "test"
    g.loc[folds[1][1], "split"] = "dev"
    g["dataset"] = GROUPED
    os.makedirs(gdir, exist_ok=True)
    for sp in ("train", "dev", "test"):
        g[g["split"] == sp].to_csv(os.path.join(gdir, f"{sp}.csv"), index=False, encoding="utf-8")
    print("grouped split written:", g["split"].value_counts().to_dict())
load_all()


def video_overlap(df, name):
    real = df[~df["video"].str.startswith("novideo-")]
    vids = {sp: set(real.loc[real["split"] == sp, "video"]) for sp in ("train", "dev", "test")}
    te = real[real["split"] == "test"]
    shares = (df[df["split"] == "test"]["label"].astype(int).value_counts(normalize=True) * 100).round(1)
    return {"split": name, "comments": len(df), "videos": int(real["video"].nunique()),
            "test neg / neu / pos (%)": " / ".join(str(shares.get(k, 0.0)) for k in (0, 1, 2)),
            "comments without video id": int((df["video"].str.startswith("novideo-")).sum()),
            "test comments": int((df["split"] == "test").sum()),
            "test comments whose video is in train (%)": round(100 * te["video"].isin(vids["train"]).sum() / max(1, (df["split"] == "test").sum()), 1),
            "test videos also in train (%)": round(100 * len(vids["test"] & vids["train"]) / max(1, len(vids["test"])), 1),
            "videos shared by any two splits": len((vids["train"] & vids["test"]) | (vids["train"] & vids["dev"]) | (vids["dev"] & vids["test"]))}


grp = pd.concat([pd.read_csv(os.path.join(gdir, f"{sp}.csv"), keep_default_na=False, dtype=str)
                 for sp in ("train", "dev", "test")], ignore_index=True)
OVERLAP = pd.DataFrame([video_overlap(yt_all, "random"), video_overlap(grp, "grouped by video")])
assert OVERLAP.loc[1, "videos shared by any two splits"] == 0
dist = pd.crosstab(grp["split"], grp["label"], normalize="index").round(3)
print(OVERLAP.to_string(index=False))
print("\nclass shares per split (grouped):\n", dist)

## 5. Stage A — classical models on the grouped split (CPU)

Same pipelines, grids and selection on development macro-F1 as notebook 02.

In [ ]:
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression


def char_tfidf():
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True, max_features=300_000)


def make_classical(name, **hp):
    norm = ("norm", FunctionTransformer(normalise_batch))
    if name == "nb_char":
        return Pipeline([norm, ("tfidf", char_tfidf()), ("clf", ComplementNB(alpha=hp["alpha"]))])
    if name == "svm_char":
        return Pipeline([norm, ("tfidf", char_tfidf()), ("clf", LinearSVC(C=hp["C"], class_weight="balanced"))])
    if name == "lr_word_char":
        feats = FeatureUnion([
            ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=200_000,
                                     token_pattern=r"(?u)\b\w+\b")),
            ("char", char_tfidf())])
        return Pipeline([norm, ("feats", feats),
                         ("clf", LogisticRegression(C=hp["C"], max_iter=3000, class_weight="balanced"))])
    raise ValueError(name)


def grid(g):
    keys = list(g)
    for vals in __import__("itertools").product(*(g[k] for k in keys)):
        yield dict(zip(keys, vals))



if "A" in RUN_STAGES and GROUPED in DATA:
    ds, parts = GROUPED, DATA[GROUPED]
    tr, dv, te = parts["train"], parts["dev"], parts["test"]
    for name, g in CLASSICAL_GRIDS.items():
        if os.path.exists(os.path.join(run_dir(ds, name, 0, RUNS_EXT_DIR), "metrics.json")):
            print(f"[done] {ds} / {name}")
            continue
        best = None
        for hp in grid(g):
            m = make_classical(name, **hp).fit(tr["text"].tolist(), tr["label"])
            f = scores(dv["label"], m.predict(dv["text"].tolist()))["macro_f1"]
            if best is None or f > best[0]:
                best = (f, hp, m)
        f_dev, hp, m = best
        pred = m.predict(te["text"].tolist())
        metrics = {"dataset": ds, "model": name, "seed": 0, "hyperparameters": hp, "dev_macro_f1": f_dev,
                   **scores(te["label"], pred)}
        save_run(ds, name, 0, metrics, pred, root=RUNS_EXT_DIR)
        print(f"{ds:16s} {name:13s} {hp} dev F1 {f_dev:.3f} | test F1 {metrics['macro_f1']:.3f} acc {metrics['accuracy']:.3f}")
else:
    print("stage A skipped")

## 6. Stage G — transformer runs (GPU)

The fine-tuning function is the one of notebook 02. Jobs, in priority order: grouped split seed 13; seeds 42 and 87 on
TWIFIL (cleaned) and YouTube, saved in half precision for stage Q; grouped split seeds 42 and 87.

In [ ]:
if torch is not None:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    from transformers.utils import logging as hf_logging
    from torch.utils.data import DataLoader
    hf_logging.set_verbosity_error()
    hf_logging.disable_progress_bar()


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_max_len(tok, texts):
    lens = [len(x) for x in tok(list(texts), truncation=False)["input_ids"]]
    return int(min(MAX_LEN_CAP, max(16, np.percentile(lens, 99))))


def encode(tok, df, max_len):
    enc = tok(df["text"].tolist(), truncation=True, max_length=max_len)
    keys = [k for k in ("input_ids", "attention_mask", "token_type_ids") if k in enc]
    return [{**{k: enc[k][i] for k in keys}, "labels": int(y)} for i, y in enumerate(df["label"])]


def loader(tok, feats, shuffle, seed=0):
    g = torch.Generator().manual_seed(seed)
    return DataLoader(feats, batch_size=BATCH_SIZE if shuffle else 128, shuffle=shuffle, generator=g,
                      collate_fn=lambda b: tok.pad(b, return_tensors="pt"))


@torch.no_grad() if torch is not None else (lambda f: f)
def predict_loader(model, dl):
    model.eval()
    out = []
    for b in dl:
        b = {k: v.to(DEVICE) for k, v in b.items() if k != "labels"}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            out += model(**b).logits.argmax(-1).tolist()
    return out



def train_one(ds, name, seed, save_dir=None, runs_root=None):
    cfg, parts = TRANSFORMERS[name], DATA[ds]
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(cfg["hf"])
    model = AutoModelForSequenceClassification.from_pretrained(cfg["hf"], num_labels=N_LABELS[ds]).to(DEVICE)
    max_len = pick_max_len(tok, parts["train"]["text"])
    tr = loader(tok, encode(tok, parts["train"], max_len), True, seed)
    dv = loader(tok, encode(tok, parts["dev"], max_len), False)
    te = loader(tok, encode(tok, parts["test"], max_len), False)

    no_decay = ("bias", "LayerNorm.weight", "layer_norm.weight")
    groups = [{"params": [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)], "weight_decay": WEIGHT_DECAY},
              {"params": [p for n, p in model.named_parameters() if any(k in n for k in no_decay)], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(groups, lr=cfg["lr"])
    total = EPOCHS * len(tr)
    warm = int(WARMUP_RATIO * total)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / max(1, warm) if s < warm else max(0.0, (total - s) / max(1, total - warm)))
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")

    best_f1, best_state, best_epoch, bad, history = -1.0, None, 0, 0, []
    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for b in tr:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
                loss = model(**b).loss
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
        f_dev = scores(parts["dev"]["label"], predict_loader(model, dv))["macro_f1"]
        history.append({"epoch": epoch, "dev_macro_f1": f_dev})
        if f_dev > best_f1:
            best_f1, best_epoch, bad = f_dev, epoch, 0
            best_state = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad > PATIENCE:
                break
    model.load_state_dict(best_state)
    pred = predict_loader(model, te)
    metrics = {"dataset": ds, "model": name, "seed": seed, "hf_id": cfg["hf"], "lr": cfg["lr"], "max_len": max_len,
               "best_epoch": best_epoch, "dev_macro_f1": best_f1, "history": history,
               "train_minutes": (time.time() - t0) / 60, "device": GPU_NAME or DEVICE, **scores(parts["test"]["label"], pred)}
    if save_dir:                                   # half precision halves the Drive space; loaded back as fp32
        model.half().save_pretrained(save_dir, safe_serialization=True)
        tok.save_pretrained(save_dir)
        with open(os.path.join(save_dir, "bench_meta.json"), "w") as f:
            json.dump({"max_len": max_len, "pad_id": tok.pad_token_id, "pad_token": tok.pad_token,
                       "stored_dtype": "float16"}, f)
    save_run(ds, name, seed, metrics, pred, root=runs_root)
    del model, opt, best_state
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return metrics



def extra_dir(ds, name, seed):
    return os.path.join(MODELS_EXT_DIR, ds, name, f"seed{seed}", "hf")


GPU_JOBS = ([(GROUPED, m, GROUPED_SEEDS[0], False) for m in TRANSFORMERS]
            + [(ds, m, s, True) for ds in INT8_DATASETS for s in EXTRA_SEEDS for m in TRANSFORMERS]
            + [(GROUPED, m, s, False) for s in GROUPED_SEEDS[1:] for m in TRANSFORMERS])
todo_g = [j for j in GPU_JOBS if j[0] in DATA and not os.path.exists(os.path.join(run_dir(j[0], j[1], j[2], RUNS_EXT_DIR), "metrics.json"))]
print(f"{len(todo_g)} GPU runs to do")
if "G" in RUN_STAGES:
    if DEVICE != "cuda" and not ALLOW_CPU_TRAINING:
        print("Stage G needs a T4 runtime (Runtime > Change runtime type). Skipping.")
    else:
        for i, (ds, name, seed, save) in enumerate(todo_g, 1):
            hours = (time.time() - SESSION_START) / 3600
            if hours > GPU_TIME_BUDGET_H:
                print(f"Time budget reached ({hours:.2f} h). {len(todo_g) - i + 1} runs left: run again tomorrow.")
                break
            sdir = extra_dir(ds, name, seed) if save else None
            if sdir:
                os.makedirs(sdir, exist_ok=True)
            m = train_one(ds, name, seed, save_dir=sdir, runs_root=RUNS_EXT_DIR)
            print(f"[{i}/{len(todo_g)}] {ds:16s} {name:13s} seed {seed:3d} | test F1 {m['macro_f1']:.3f} acc {m['accuracy']:.3f} "
                  f"| best epoch {m['best_epoch']} | {m['train_minutes']:.1f} min" + (" | saved (fp16)" if save else ""))

## 7. Export helpers and links to the benchmark files of notebook 02

In [ ]:
def dir_size_mb(path, files=None):
    if os.path.isfile(path):
        return os.path.getsize(path) / 2 ** 20
    names = files if files is not None else os.listdir(path)
    return sum(os.path.getsize(os.path.join(path, n)) for n in names if os.path.isfile(os.path.join(path, n))) / 2 ** 20


class _Logits(torch.nn.Module if torch is not None else object):
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kw = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kw["token_type_ids"] = token_type_ids
        return self.m(**kw).logits


def export_onnx(hf_dir, out_path):
    tok = AutoTokenizer.from_pretrained(hf_dir)
    model = AutoModelForSequenceClassification.from_pretrained(hf_dir).float().eval()
    enc = tok(["مثال قصير", "exemple un peu plus long pour la forme"], padding=True, return_tensors="pt")
    names = [k for k in ("input_ids", "attention_mask", "token_type_ids") if k in enc]
    args = tuple(enc[k] for k in names)
    dyn = {k: {0: "batch", 1: "seq"} for k in names}
    dyn["logits"] = {0: "batch"}
    kw = dict(input_names=names, output_names=["logits"], dynamic_axes=dyn, opset_version=17)
    wrapper = _Logits(model).eval()      # a new Module starts in train mode; the exporter would restore that mode
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            torch.onnx.export(wrapper, args, out_path, dynamo=False, **kw)
    except TypeError:                                   # older torch without the dynamo argument
        torch.onnx.export(wrapper, args, out_path, **kw)
    except Exception as e:                              # future torch without the legacy exporter
        print("legacy ONNX exporter failed, using the dynamo exporter:", str(e)[:150])
        try:
            import onnxscript  # noqa: F401
        except ImportError:
            _pip("onnxscript")
        b, q = torch.export.Dim("batch"), torch.export.Dim("seq", max=512)
        shapes = tuple({0: b, 1: q} for _ in names)
        torch.onnx.export(wrapper, args, out_path, dynamo=True, input_names=names, output_names=["logits"],
                          dynamic_shapes=shapes, opset_version=18)
    model.eval()
    return tok, model, names


def check_equivalence(hf_dir, onnx_path, tok, model, texts, meta):
    import onnxruntime as ort
    from tokenizers import Tokenizer
    tk = Tokenizer.from_file(os.path.join(hf_dir, "tokenizer.json"))
    tk.enable_truncation(max_length=meta["max_len"])
    tk.enable_padding(pad_id=meta["pad_id"], pad_token=meta["pad_token"])
    encs = tk.encode_batch(texts)
    ref = tok(texts, truncation=True, max_length=meta["max_len"], padding=True, return_tensors="pt")
    assert [e.ids for e in encs] == ref["input_ids"].tolist(), "tokenizers vs AutoTokenizer ids differ"
    sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    feed = {"input_ids": np.array([e.ids for e in encs], dtype=np.int64),
            "attention_mask": np.array([e.attention_mask for e in encs], dtype=np.int64),
            "token_type_ids": np.array([e.type_ids for e in encs], dtype=np.int64)}
    got = sess.run(None, {i.name: feed[i.name] for i in sess.get_inputs()})[0]
    with torch.inference_mode():
        want = model(**ref).logits.numpy()
    return float(np.abs(got - want).max())



VARIANTS = {"ONNX int8": dict(per_channel=False, reduce_range=False),
            "ONNX int8 per-channel (no RR)": dict(per_channel=True, reduce_range=False),
            "ONNX int8 per-channel+RR": dict(per_channel=True, reduce_range=True)}
ORIG_FILE = {"PyTorch fp32": "PyTorch_fp32", "ONNX fp32": "ONNX_fp32", "ONNX int8": "ONNX_int8",
             "ONNX int8 per-channel (no RR)": "ONNX_int8_per-channel", "ONNX int8 per-channel+RR": "ONNX_int8_per-channel+RR"}


def orig_bench(ds, name, fmt):
    p = os.path.join(BENCH_DIR, ds, f"{name}__{ORIG_FILE.get(fmt, fmt.replace(' ', '_'))}.json")
    return json.load(open(p, encoding="utf-8")) if os.path.exists(p) else None


def texts_file(ds):
    p = os.path.join(BENCH_EXT_DIR, ds, "_test_texts.json")
    os.makedirs(os.path.dirname(p), exist_ok=True)
    if not os.path.exists(p):
        with open(p, "w", encoding="utf-8") as f:
            json.dump(DATA[ds]["test"]["text"].tolist(), f, ensure_ascii=False)
    return p


def onnx_spec(path, hf_dir):
    meta = json.load(open(os.path.join(hf_dir, "bench_meta.json")))
    return {"kind": "onnx", "path": path, "tokenizer": os.path.join(hf_dir, "tokenizer.json"), **meta}


def thread_check():
    """Predictions of the seed-13 int8 models of notebook 02 with EVAL_THREADS must equal the stored one-thread predictions."""
    n_threads = EVAL_THREADS or os.cpu_count()
    if n_threads == 1:
        return 1
    checked = 0
    for ds in INT8_DATASETS:
        for name in TRANSFORMERS:
            hf = os.path.join(MODELS_DIR, ds, name, "hf")
            for fmt, fn in (("ONNX int8", "model.int8.onnx"), ("ONNX int8 per-channel+RR", "model.int8pcrr.onnx")):
                ref, path = orig_bench(ds, name, fmt), os.path.join(MODELS_DIR, ds, name, fn)
                if ref is None or not os.path.exists(path):
                    continue
                r = run_worker(onnx_spec(path, hf), texts_file(ds), os.path.join(BENCH_EXT_DIR, ds, "_tcheck.json"),
                               "predict", n_threads, ["--limit", CHECK_N, "--batch", BENCH_BATCH])
                same = r["preds"] == ref["preds"][:CHECK_N]
                print(f"thread check ({n_threads} threads vs 1): {ds} / {name} / {fmt}: {'identical' if same else 'DIFFERENT'}")
                if not same:
                    print("predictions depend on the number of threads: stage Q uses one thread")
                    return 1
                checked += 1
            if checked >= 2:
                break                                     # one model per dataset is enough
    if not checked:
        print("thread check not possible (no seed-13 int8 files); using one thread")
        return 1
    return n_threads


def q_out(ds, name, seed):
    return os.path.join(BENCH_EXT_DIR, "int8_seeds", ds, f"{name}__seed{seed}.json")

## 8. Stage R — standardised CPU benchmark

Every artefact of the paper (3 classical models and 4 transformers in 4 formats, on TWIFIL cleaned and YouTube) is
copied to local disk; the ONNX fp32 graphs are exported again from the saved PyTorch models. Each artefact is first
checked against the paper: its predictions for the first test texts must equal the stored ones. Then `BENCH_PASSES`
identical passes visit all artefacts in a new random order each time; every pass is a fresh process that measures cold
start, single-text latency (three rounds of 300 texts after 20 warm-up calls) and the peak memory of that serving process.
Batch-scoring memory comes from one identical prediction run per artefact (the whole test set in batches of 32).

In [ ]:
def copy_if_missing(src, dst):
    if os.path.isdir(src):
        if not os.path.exists(dst):
            shutil.copytree(src, dst)
    elif not os.path.exists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copyfile(src, dst)
    return dst


def prepare_artefacts(ds):
    root, out = os.path.join(LOCAL_ARTEFACTS, ds), []
    for name in CLASSICAL_GRIDS:
        src = os.path.join(MODELS_DIR, ds, name, "model.joblib")
        if os.path.exists(src):
            p = copy_if_missing(src, os.path.join(root, name, "model.joblib"))
            out.append({"model": name, "format": "scikit-learn", "kind": "sklearn", "path": p, "size_mb": dir_size_mb(p)})
    for name in TRANSFORMERS:
        src_hf = os.path.join(MODELS_DIR, ds, name, "hf")
        if not os.path.exists(os.path.join(src_hf, "bench_meta.json")):
            print(f"missing model: {ds} / {name}")
            continue
        hf = copy_if_missing(src_hf, os.path.join(root, name, "hf"))
        meta = json.load(open(os.path.join(hf, "bench_meta.json")))
        tok_json = os.path.join(hf, "tokenizer.json")
        out.append({"model": name, "format": "PyTorch fp32", "kind": "torch", "path": hf, "size_mb": dir_size_mb(hf), **meta})
        fp32 = os.path.join(root, name, "model.onnx")
        if not os.path.exists(fp32):
            tok, model, names = export_onnx(hf, fp32)
            diff = check_equivalence(hf, fp32, tok, model, DATA[ds]["dev"]["text"].head(16).tolist(), meta)
            assert diff < 1e-3, f"{ds}/{name}: ONNX fp32 differs from PyTorch (max |Δ| = {diff:.2e})"
            del model
        for fmt, fn in (("ONNX fp32", "model.onnx"), ("ONNX int8", "model.int8.onnx"), ("ONNX int8 per-channel+RR", "model.int8pcrr.onnx")):
            p = os.path.join(root, name, fn) if fn == "model.onnx" else copy_if_missing(os.path.join(MODELS_DIR, ds, name, fn), os.path.join(root, name, fn))
            out.append({"model": name, "format": fmt, "kind": "onnx", "path": p, "tokenizer": tok_json,
                        "size_mb": dir_size_mb(p) + dir_size_mb(tok_json), **meta})
    return out


def r_path(ds, a, tag):
    os.makedirs(os.path.join(BENCH_R_DIR, ds), exist_ok=True)
    return os.path.join(BENCH_R_DIR, ds, f"{a['model']}__{a['format'].replace(' ', '_')}.{tag}.json")


ARTEFACTS = {}
if "R" in RUN_STAGES:
    if torch is None:
        raise RuntimeError("PyTorch is needed for the ONNX export")
    if STRICT_CPU and not paper_cpu():
        raise RuntimeError(f"Stage R needs the paper's processor ({CPU_MODEL or 'AVX2 without AVX-512/VNNI'}); this session has "
                           f"{cpu_name()} with {cpu_flags()}. Runtime > Disconnect and delete runtime, connect again, and re-run.")
    for ds in [d for d in BENCH_DATASETS if d in DATA]:
        ARTEFACTS[ds] = prepare_artefacts(ds)
        tp = texts_file(ds)
        for a in ARTEFACTS[ds]:
            # 1) same artefact as in notebook 02? 2) one batch-scoring run for memory
            chk = r_path(ds, a, "check")
            if not os.path.exists(chk):
                ref = orig_bench(ds, a["model"], a["format"])
                r = run_worker(a, tp, chk + ".run", "predict", 1, ["--limit", CHECK_N, "--batch", BENCH_BATCH])
                r["matches_paper"] = None if ref is None else r["preds"] == ref["preds"][:CHECK_N]
                r["artefact"] = {k: v for k, v in a.items() if k != "path"}
                r.update({"cpu": cpu_name(), "cpu_flags": cpu_flags()})
                with open(chk, "w") as f:
                    json.dump(r, f)
            batch = r_path(ds, a, "batch")
            if MEASURE_BATCH_MEMORY and not os.path.exists(batch):
                r = run_worker(a, tp, batch + ".run", "predict", BENCH_THREADS, ["--batch", BENCH_BATCH])
                r.update(scores(DATA[ds]["test"]["label"], r["preds"]))
                r.update({"cpu": cpu_name(), "cpu_flags": cpu_flags()})
                with open(batch, "w") as f:
                    json.dump(r, f)
            c = json.load(open(chk))
            print(f"{ds:13s} {DISPLAY[a['model']]:13s} {a['format']:25s} same predictions as notebook 02: {c['matches_paper']}")
    items = [(ds, a) for ds in ARTEFACTS for a in ARTEFACTS[ds]]
    for p in range(1, BENCH_PASSES + 1):
        order = items[:]
        random.Random(p).shuffle(order)
        n_new, t0 = 0, time.time()
        for ds, a in order:
            out = r_path(ds, a, f"p{p}")
            if os.path.exists(out):
                continue
            r = run_worker(a, texts_file(ds), out + ".run", "timing", BENCH_THREADS,
                           ["--n_single", N_SINGLE, "--warmup", WARMUP_CALLS, "--rounds", BENCH_ROUNDS])
            r.update({"pass": p, "cpu": cpu_name(), "cpu_flags": cpu_flags(),
                      "finished_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")})
            with open(out, "w") as f:
                json.dump(r, f)
            n_new += 1
        print(f"pass {p}/{BENCH_PASSES}: {n_new} artefacts timed in {(time.time() - t0) / 60:.1f} min")
else:
    print("stage R skipped")

## 9. Stage Q — int8 across seeds (CPU)

For every saved seed-42 and seed-87 model: export to ONNX fp32 (with the equivalence checks of notebook 02), derive the
three int8 variants of the paper (default; per-channel without reduce-range; per-channel with reduce-range), and predict
the whole test set in batches of 32 in test order, as in the paper. Seed 13 is read from the benchmark files of notebook 02.
Before the first evaluation, the notebook checks that the number of threads does not change the predictions. Models
that stage G has not trained yet are skipped and picked up by a later run.

In [ ]:
q_todo = [(ds, n, s) for ds in INT8_DATASETS if ds in DATA for n in TRANSFORMERS for s in EXTRA_SEEDS
          if not os.path.exists(q_out(ds, n, s))]
print(f"stage Q: {len(q_todo)} seed models to evaluate")
if "Q" in RUN_STAGES and q_todo:
    if torch is None:
        raise RuntimeError("PyTorch is needed for the ONNX export")
    if STRICT_CPU and not paper_cpu():
        raise RuntimeError(f"Stage Q needs the paper's processor ({CPU_MODEL or 'AVX2 without AVX-512/VNNI'}); this session has "
                           f"{cpu_name()} with {cpu_flags()}. Runtime > Disconnect and delete runtime, connect again, and re-run.")
    from onnxruntime.quantization import quantize_dynamic, QuantType
    q_threads = thread_check()
    for ds, name, seed in q_todo:
        hf = extra_dir(ds, name, seed)
        if not os.path.exists(os.path.join(hf, "bench_meta.json")):
            print(f"[wait] {ds} / {name} / seed {seed}: model not trained yet (stage G)")
            continue
        tmp = os.path.join(LOCAL_ARTEFACTS, "_q_tmp")
        shutil.rmtree(tmp, ignore_errors=True)
        os.makedirs(tmp)
        t0 = time.time()
        meta = json.load(open(os.path.join(hf, "bench_meta.json")))
        fp32 = os.path.join(tmp, "model.onnx")
        tok, model, names = export_onnx(hf, fp32)
        diff = check_equivalence(hf, fp32, tok, model, DATA[ds]["dev"]["text"].head(16).tolist(), meta)
        assert diff < 1e-3, f"{ds}/{name}/seed {seed}: ONNX fp32 differs from PyTorch (max |Δ| = {diff:.2e})"
        del model
        res = {"dataset": ds, "model": name, "seed": seed, "max_abs_logit_diff": diff, "threads": q_threads,
               "cpu": cpu_name(), "cpu_flags": cpu_flags(), "versions": VERSIONS, "variants": {}}
        paths = {"ONNX fp32": fp32}
        for fmt, kw in VARIANTS.items():
            paths[fmt] = os.path.join(tmp, fmt.replace(" ", "_") + ".onnx")
            quantize_dynamic(fp32, paths[fmt], weight_type=QuantType.QInt8, **kw)
        for fmt, path in paths.items():
            r = run_worker(onnx_spec(path, hf), texts_file(ds), os.path.join(tmp, "pred.json"), "predict", q_threads,
                           ["--batch", BENCH_BATCH])
            res["variants"][fmt] = {"preds": r["preds"], **scores(DATA[ds]["test"]["label"], r["preds"])}
        os.makedirs(os.path.dirname(q_out(ds, name, seed)), exist_ok=True)
        with open(q_out(ds, name, seed), "w") as f:
            json.dump(res, f)
        shutil.rmtree(tmp, ignore_errors=True)
        if DELETE_EXTRA_WEIGHTS_AFTER_USE:
            shutil.rmtree(os.path.dirname(hf), ignore_errors=True)
        v = res["variants"]
        print(f"{ds:13s} {DISPLAY[name]:13s} seed {seed} | fp32 {100 * v['ONNX fp32']['macro_f1']:.1f} | "
              + " | ".join(f"{k.replace('ONNX int8', 'int8')} {100 * (v[k]['macro_f1'] - v['ONNX fp32']['macro_f1']):+.1f}" for k in VARIANTS)
              + f" | {(time.time() - t0) / 60:.1f} min")

## 10. Stage T — tables

In [ ]:
def df_to_md(df):
    cols = [str(c) for c in df.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    lines += ["| " + " | ".join("" if (isinstance(v, float) and math.isnan(v)) else str(v) for v in r) + " |"
              for r in df.itertuples(index=False)]
    return "\n".join(lines) + "\n"


def save_table(df, name):
    df.to_csv(os.path.join(REPORT_EXT_DIR, f"{name}.csv"), index=False)
    with open(os.path.join(REPORT_EXT_DIR, f"{name}.md"), "w", encoding="utf-8") as f:
        f.write(df_to_md(df))
    print(f"\n{name}\n" + df.to_string(index=False))


def pm(v):
    v = [x for x in v if x == x]
    if not v:
        return ""
    return f"{100 * np.mean(v):.1f} ± {100 * np.std(v, ddof=1):.1f}" if len(v) > 1 else f"{100 * v[0]:.1f}"


def runs_of(root, ds, name):
    return [json.load(open(p)) for p in sorted(glob.glob(os.path.join(root, ds, name, "seed*", "metrics.json")))]


def paired_diff_ci(y, pa, pb, n=N_BOOTSTRAP, seed=0):
    rng = np.random.default_rng(seed)
    y, pa, pb = map(np.asarray, (y, pa, pb))
    labs = sorted(set(y))
    f = lambda idx, p: f1_score(y[idx], p[idx], labels=labs, average="macro", zero_division=0)
    d = [f(idx, pa) - f(idx, pb) for idx in (rng.integers(0, len(y), len(y)) for _ in range(n))]
    return np.percentile(d, [2.5, 97.5])


save_table(OVERLAP, "table_youtube_video_overlap")

# ---- T1: random vs video-grouped YouTube split ----
rows = []
for name in list(CLASSICAL_GRIDS) + list(TRANSFORMERS):
    a, b = runs_of(RUNS_DIR, "youtube", name), runs_of(RUNS_EXT_DIR, GROUPED, name)
    if not b:
        continue
    fa, fb = [r["macro_f1"] for r in a], [r["macro_f1"] for r in b]
    rows.append({"model": DISPLAY[name], "seeds (random / grouped)": f"{len(a)} / {len(b)}",
                 "macro-F1 random (%)": pm(fa), "macro-F1 grouped (%)": pm(fb),
                 "Δ grouped − random (pts)": round(100 * (np.mean(fb) - np.mean(fa)), 1) if fa else np.nan,
                 "accuracy random (%)": pm([r["accuracy"] for r in a]), "accuracy grouped (%)": pm([r["accuracy"] for r in b]),
                 "neutral F1 grouped (%)": round(100 * np.mean([r["per_class_f1"]["neutral"] for r in b]), 1)})
if rows:
    save_table(pd.DataFrame(rows), "table_grouped_split")
    runs_long = []
    for root, ds, label in ((RUNS_DIR, "youtube", "random"), (RUNS_EXT_DIR, GROUPED, "grouped")):
        for name in list(CLASSICAL_GRIDS) + list(TRANSFORMERS):
            for r in runs_of(root, ds, name):
                runs_long.append({"split": label, "model": DISPLAY[name], "seed": r["seed"],
                                  "macro-F1 (%)": round(100 * r["macro_f1"], 2), "accuracy (%)": round(100 * r["accuracy"], 2),
                                  **{f"F1 {k} (%)": round(100 * v, 2) for k, v in r["per_class_f1"].items()},
                                  "best epoch": r.get("best_epoch"), "hyper-parameters": json.dumps(r.get("hyperparameters", ""))})
    save_table(pd.DataFrame(runs_long), "table_grouped_runs")
    # paired bootstrap on the grouped test set: best transformer vs best classical (first seed)
    preds = {}
    for name in list(CLASSICAL_GRIDS) + list(TRANSFORMERS):
        p = os.path.join(run_dir(GROUPED, name, 0 if name in CLASSICAL_GRIDS else GROUPED_SEEDS[0], RUNS_EXT_DIR), "test_preds.csv")
        if os.path.exists(p):
            preds[name] = pd.read_csv(p)
    cls = [m for m in preds if m in CLASSICAL_GRIDS]
    trf = [m for m in preds if m in TRANSFORMERS]
    if cls and trf:
        f1 = {m: f1_score(d["label"], d["pred"], average="macro") for m, d in preds.items()}
        bt, bc = max(trf, key=f1.get), max(cls, key=f1.get)
        lo, hi = paired_diff_ci(preds[bt]["label"], preds[bt]["pred"], preds[bc]["pred"])
        save_table(pd.DataFrame([{"dataset": GROUPED, "comparison": f"{DISPLAY[bt]} (seed {GROUPED_SEEDS[0]}) − {DISPLAY[bc]}",
                                  "Δ macro-F1 (pts)": round(100 * (f1[bt] - f1[bc]), 1),
                                  "95% CI": f"[{100 * lo:.1f}, {100 * hi:.1f}]"}]), "table_grouped_bootstrap_ci")

# ---- T2: int8 across seeds ----
long = []
for ds in INT8_DATASETS:
    if ds not in DATA:
        continue
    y = DATA[ds]["test"]["label"].values
    for name in TRANSFORMERS:
        for seed in [SEEDS[0]] + EXTRA_SEEDS:
            if seed == SEEDS[0]:
                vs = {fmt: orig_bench(ds, name, fmt) for fmt in ["ONNX fp32", *VARIANTS]}
                vs = {k: v for k, v in vs.items() if v is not None}
                vs = {k: {"preds": v["preds"], **scores(y, v["preds"])} for k, v in vs.items()}
            elif os.path.exists(q_out(ds, name, seed)):
                vs = json.load(open(q_out(ds, name, seed)))["variants"]
            else:
                continue
            if "ONNX fp32" not in vs:
                continue
            ref = vs["ONNX fp32"]["macro_f1"]
            for fmt, v in vs.items():
                long.append({"dataset": ds, "model": DISPLAY[name], "seed": seed, "format": fmt,
                             "macro-F1 (%)": round(100 * v["macro_f1"], 2), "Δ vs fp32 (pts)": round(100 * (v["macro_f1"] - ref), 2),
                             "constant predictions": len(set(v["preds"])) == 1})
if long:
    LONG = pd.DataFrame(long)
    save_table(LONG, "table_int8_seeds_long")
    summ = []
    for (ds, model), g in LONG.groupby(["dataset", "model"], sort=False):
        row = {"dataset": ds, "model": model}
        f = g[g["format"] == "ONNX fp32"]["macro-F1 (%)"]
        row["seeds"] = len(f)
        row["fp32 macro-F1 (%)"] = f"{f.mean():.1f} ± {f.std(ddof=1):.1f}" if len(f) > 1 else (f"{f.iloc[0]:.1f}" if len(f) else "")
        for fmt, short in (("ONNX int8", "default"), ("ONNX int8 per-channel (no RR)", "PC, no RR"),
                           ("ONNX int8 per-channel+RR", "PC+RR")):
            d = g[g["format"] == fmt]
            x = d["Δ vs fp32 (pts)"]
            row[f"Δ {short} (pts)"] = (f"{x.mean():+.1f} ± {x.std(ddof=1):.1f}" if len(x) > 1 else (f"{x.iloc[0]:+.1f}" if len(x) else ""))
            row[f"Δ {short} range"] = f"{x.min():+.1f} to {x.max():+.1f}" if len(x) else ""
            row[f"{short}: seeds / constant"] = f"{len(x)} / {int(d['constant predictions'].sum())}"
        summ.append(row)
    save_table(pd.DataFrame(summ), "table_int8_seeds")

# ---- T3: standardised deployment table ----
dep = []
for ds in BENCH_DATASETS:
    if ds not in DATA:
        continue
    for p in sorted(glob.glob(os.path.join(BENCH_R_DIR, ds, "*.check.json"))):
        chk = json.load(open(p))
        stem = p[: -len(".check.json")]
        passes = [json.load(open(t)) for t in sorted(glob.glob(stem + ".p[0-9]*.json"))]
        model, fmt = chk["artefact"]["model"], chk["artefact"]["format"]
        if not passes:
            continue
        if os.path.exists(stem + ".batch.json"):
            b = json.load(open(stem + ".batch.json"))
            b_rss = b["batch_peak_rss_mb"]
        else:                                             # main pass of notebook 02: batch scoring, measured once
            b = orig_bench(ds, model, fmt) or {}
            b_rss = b.get("peak_rss_mb", np.nan)
        q = scores(DATA[ds]["test"]["label"], b["preds"]) if "preds" in b else {"macro_f1": np.nan, "accuracy": np.nan}
        p50 = [x["p50_ms"] for x in passes]
        dep.append({"dataset": ds, "model": model, "format": fmt, "macro_f1": q["macro_f1"], "accuracy": q["accuracy"],
                    "size_mb": chk["artefact"]["size_mb"], "passes": len(passes),
                    "p50_ms": float(np.median(p50)), "p95_ms": float(np.median([x["p95_ms"] for x in passes])),
                    "p50_min": min(p50), "p50_max": max(p50),
                    "cold_s": float(np.median([x["load_s"] for x in passes])),
                    "serving_rss": float(np.median([x["serving_peak_rss_mb"] for x in passes])),
                    "serving_rss_range": max(x["serving_peak_rss_mb"] for x in passes) - min(x["serving_peak_rss_mb"] for x in passes),
                    "batch_rss": b_rss, "matches_paper": chk.get("matches_paper")})
DEP = pd.DataFrame(dep)
if len(DEP):
    nat = DEP[DEP["format"].isin(["scikit-learn", "PyTorch fp32"])].set_index(["dataset", "model"])
    DEP["speedup"] = [nat["p50_ms"].get((r.dataset, r.model), np.nan) / r.p50_ms if r.model in TRANSFORMERS else np.nan
                      for r in DEP.itertuples()]
    DEP["delta"] = [r.macro_f1 - nat["macro_f1"].get((r.dataset, r.model), np.nan) for r in DEP.itertuples()]
    T3 = pd.DataFrame({
        "dataset": DEP["dataset"], "model": DEP["model"].map(DISPLAY), "format": DEP["format"],
        "macro-F1 (%)": (100 * DEP["macro_f1"]).round(1), "Δ vs native (pts)": (100 * DEP["delta"]).round(2),
        "size (MiB)": DEP["size_mb"].round(1), "serving peak RSS (MiB)": DEP["serving_rss"].round(0),
        "batch peak RSS (MiB)": DEP["batch_rss"].round(0), "cold start (s)": DEP["cold_s"].round(2),
        "p50 (ms)": DEP["p50_ms"].round(2), "p95 (ms)": DEP["p95_ms"].round(2),
        "p50 range across passes (ms)": [f"{a:.2f}–{b:.2f}" for a, b in zip(DEP["p50_min"], DEP["p50_max"])],
        "p50 spread (% of median)": (100 * (DEP["p50_max"] - DEP["p50_min"]) / DEP["p50_ms"]).round(1),
        "passes": DEP["passes"], "speed-up vs PyTorch fp32": DEP["speedup"].round(2),
        "same predictions as notebook 02": DEP["matches_paper"]}).sort_values(["dataset", "p50 (ms)"])
    save_table(T3, "table_deployment_standard")

    # rank agreement with the benchmark of notebook 02
    agree = []
    old = pd.read_csv(os.path.join(PROJECT_DIR, "reports" + suffix, "table_deployment.csv")) \
        if os.path.exists(os.path.join(PROJECT_DIR, "reports" + suffix, "table_deployment.csv")) else None
    if old is not None:
        for ds in BENCH_DATASETS:
            n = T3[T3["dataset"] == ds][["model", "format", "p50 (ms)"]]
            o = old[(old["dataset"] == ds)][["model", "format", "p50 (ms)"]]
            m = n.merge(o, on=["model", "format"], suffixes=(" new", " nb02"))
            if len(m) > 2:
                from scipy.stats import spearmanr
                agree.append({"dataset": ds, "artefacts": len(m),
                              "Spearman rho (p50, vs notebook 02)": round(spearmanr(m["p50 (ms) new"], m["p50 (ms) nb02"])[0], 3),
                              "median p50 ratio to notebook 02": round(float(np.median(m["p50 (ms) new"] / m["p50 (ms) nb02"])), 2)})
        if agree:
            save_table(pd.DataFrame(agree), "table_benchmark_agreement")

    # Pareto frontier and budget tiers on the standardised numbers
    def pareto_mask(cost, quality):
        cost, quality = np.asarray(cost), np.asarray(quality)
        keep = np.ones(len(cost), bool)
        for i in range(len(cost)):
            dom = (cost <= cost[i]) & (quality >= quality[i]) & ((cost < cost[i]) | (quality > quality[i]))
            keep[i] = not dom.any()
        return keep

    fr, sc = [], []
    for ds in BENCH_DATASETS:
        b = DEP[DEP["dataset"] == ds]
        for col, axis in (("p50_ms", "latency"), ("size_mb", "size")):
            for r in b[pareto_mask(b[col], b["macro_f1"])].sort_values(col).itertuples():
                fr.append({"dataset": ds, "axis": axis, "model": DISPLAY[r.model], "format": r.format,
                           "macro-F1 (%)": round(100 * r.macro_f1, 1), "p50 (ms)": round(r.p50_ms, 2), "size (MiB)": round(r.size_mb, 1)})
        for sname, (mx, p95) in SCENARIOS.items():
            ok = b[(b["size_mb"] <= mx) & (b["p95_ms"] <= p95)]
            if len(ok):
                r = ok.sort_values(["macro_f1", "p95_ms"], ascending=[False, True]).iloc[0]
                sc.append({"dataset": ds, "scenario": sname, "best configuration": f"{DISPLAY[r.model]} ({r.format})",
                           "macro-F1 (%)": round(100 * r.macro_f1, 1), "size (MiB)": round(r.size_mb, 1),
                           "p95 (ms)": round(r.p95_ms, 2), "candidates that fit": len(ok)})
    save_table(pd.DataFrame(fr), "table_pareto_frontier_standard")
    save_table(pd.DataFrame(sc), "table_scenarios_standard")

### TWIFIL: separating the link-only shortcut from the other effects of the audit

On the test set of the DziriBERT split, accuracy and macro-F1 of the models trained on that split are computed on nested
subsets: all tweets; tweets with text (link-only tweets removed); the tweets kept by the audit (also without conflicting
labels and within-split repeats); and, among those, the tweets whose normalised text never occurs in the DziriBERT training
data (no leakage). The models trained on the cleaned split are scored on the last two subsets, which they share.

In [ ]:
import html
URL_RE = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)
MENTION_RE = re.compile(r"@\w+:?")
RT_RE = re.compile(r"^\s*RT\b\s*")
HASHTAG_RE = re.compile(r"#(\w+)")
ARABIC_LETTER_RE = re.compile(r"[ء-غف-يٱ-ۓۺ-ۿ]")
LATIN_LETTER_RE = re.compile(r"[A-Za-zÀ-ɏ]")
DIACRITICS_RE = re.compile(r"[ؐ-ًؚ-ٰٟۖ-ۭ]")
EMOJI_RE = re.compile(
    "["
    "\U0001F1E6-\U0001F1FF"   # flags
    "\U0001F300-\U0001F5FF"   # symbols & pictographs
    "\U0001F600-\U0001F64F"   # emoticons
    "\U0001F680-\U0001F6FF"   # transport & map
    "\U0001F700-\U0001FAFF"   # extended pictographs
    "☀-➿"           # misc symbols, dingbats (incl. heart)
    "]"
)
EMOTICON_RE = re.compile(r"(?:[:;=][-']?[()\[\]DPp/\\|]|<3)")
AR_NORM = str.maketrans({"أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا", "ى": "ي", "ة": "ه", "ؤ": "و", "ئ": "ي", "ـ": ""})


def clean_text(text):
    s = html.unescape(str(text))
    s = RT_RE.sub(" ", s)
    s = URL_RE.sub(" ", s)
    s = MENTION_RE.sub(" ", s)
    s = HASHTAG_RE.sub(lambda m: m.group(1).replace("_", " "), s)
    return " ".join(s.split())


def has_content(text):
    """True if the cleaned text still has at least one letter, emoji or emoticon."""
    return bool(ARABIC_LETTER_RE.search(text) or LATIN_LETTER_RE.search(text)
                or EMOJI_RE.search(text) or EMOTICON_RE.search(text))


def dedup_key(text):
    """Aggressive normalisation used only for duplicate / conflict / leakage detection."""
    s = DIACRITICS_RE.sub("", text.lower()).translate(AR_NORM)
    s = re.sub(r"(.)\1{2,}", r"\1\1", s)                       # elongation: ههههه -> هه, cooool -> cool
    s = "".join(ch if (ch.isalnum() or ch.isspace() or EMOJI_RE.match(ch)) else " " for ch in s)
    s = " ".join(s.split())
    return s if s else text.strip()                              # emoticon-only texts keep their own key



if "twifil_official" in DATA and "twifil_clean" in DATA:
    off = pd.concat([pd.read_csv(os.path.join(PROC_DIR, "twifil_official", f"{sp}.csv"), keep_default_na=False, dtype={"text_raw": str})
                     for sp in ("train", "dev", "test")])
    off["key"] = off["text_raw"].map(clean_text).map(dedup_key)
    off["link_only"] = ~off["text_raw"].map(clean_text).map(has_content)
    train_keys = set(off.loc[off["split"] != "test", "key"])
    te = off[off["split"] == "test"].set_index("id")
    clean_test_ids = set(pd.read_csv(os.path.join(PROC_DIR, "twifil_clean", "test.csv"), dtype=str)["id"])
    subsets = {"all test tweets": te.index,
               "with text (link-only removed)": te.index[~te["link_only"]],
               "kept by the audit": te.index[te.index.isin(clean_test_ids)],
               "kept by the audit, text unseen in training": te.index[te.index.isin(clean_test_ids) & ~te["key"].isin(train_keys)]}
    rows = []
    for train_ds, label in (("twifil_official", "trained on DziriBERT split"), ("twifil_clean", "trained on cleaned split")):
        for name in list(CLASSICAL_GRIDS) + list(TRANSFORMERS):
            files = sorted(glob.glob(os.path.join(RUNS_DIR, train_ds, name, "seed*", "test_preds.csv")))
            if not files:
                continue
            for sub, ids in subsets.items():
                accs, f1s = [], []
                for fpath in files:
                    pr = pd.read_csv(fpath, dtype={"id": str}).set_index("id")
                    pr = pr[pr.index.isin(ids)]
                    if len(pr) < len(ids) * 0.99:            # this model was not tested on the whole subset
                        break
                    accs.append((pr["label"] == pr["pred"]).mean())
                    f1s.append(f1_score(pr["label"], pr["pred"], average="macro"))
                if accs:
                    rows.append({"training data": label, "model": DISPLAY[name], "test subset": sub, "n": len(ids),
                                 "accuracy (%)": round(100 * np.mean(accs), 1), "macro-F1 (%)": round(100 * np.mean(f1s), 1),
                                 "seeds": len(accs)})
    if rows:
        save_table(pd.DataFrame(rows), "table_twifil_decomposition")
    print({k: len(v) for k, v in subsets.items()})

## 11. Provenance

In [ ]:
prov = {"created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
        "smoke_test": SMOKE_TEST, "run_stages": RUN_STAGES, "cpu": cpu_name(), "cpu_flags": cpu_flags(),
        "paper_cpu_type": paper_cpu(), "cpu_model_required": CPU_MODEL, "stage_R_dir": os.path.basename(BENCH_R_DIR),
        "gpu": GPU_NAME, "versions": VERSIONS,
        "config": {"GROUPED_SEEDS": GROUPED_SEEDS, "EXTRA_SEEDS": EXTRA_SEEDS, "INT8_DATASETS": INT8_DATASETS,
                   "BENCH_PASSES": BENCH_PASSES, "N_SINGLE": N_SINGLE, "BENCH_ROUNDS": BENCH_ROUNDS,
                   "WARMUP_CALLS": WARMUP_CALLS, "BENCH_BATCH": BENCH_BATCH, "BENCH_THREADS": BENCH_THREADS,
                   "EVAL_THREADS": EVAL_THREADS, "SCENARIOS": {k: list(v) for k, v in SCENARIOS.items()}}}
with open(os.path.join(REPORT_EXT_DIR, "provenance_03.json"), "w") as f:
    json.dump(prov, f, indent=2, default=str)
left_g = sum(1 for j in GPU_JOBS if j[0] in DATA and not os.path.exists(os.path.join(run_dir(j[0], j[1], j[2], RUNS_EXT_DIR), "metrics.json")))
left_q = sum(1 for ds in INT8_DATASETS if ds in DATA for n in TRANSFORMERS for s in EXTRA_SEEDS if not os.path.exists(q_out(ds, n, s)))
left_r = sum(1 for ds in BENCH_DATASETS for p in range(1, BENCH_PASSES + 1)
             for a in ARTEFACTS.get(ds, []) if not os.path.exists(r_path(ds, a, f"p{p}")))
print(f"GPU runs left: {left_g} | stage Q models left: {left_q} | stage R timings left (this session's artefacts): {left_r}")
print("When everything is 0: zip the folders", REPORT_EXT_DIR, "and", BENCH_R_DIR, "and send them.")